# Multi-goal geodesic ranking on expanding square grids

Use a Colab GPU. Fit **one shared adapter** for six hand-positioned goals, three per room. Start from identity and warm-start each later checkpoint. Each checkpoint uses 500 updates with cumulative replay of labeled pairs and uniform sampling over fitting goals.

- **Each fitting goal has a fixed square grid with 2-unit spacing.** Half-widths expand through 6, 12, 18, 24, 30, 36, 42, and 46. Every newly enclosed legal intersection enters that goal’s fitting pool, including edges and corners. Shortest legal path distance determines pair order. A pair needs a gap of at least 0.5 units and can lie within one band or across bands.
- **Eight unseen goals** span the two rooms. Their coordinates and any matching rendered images are reserved from every fitting role. The held-out grid also excludes images used during fitting and goal reference images.
- **Evaluate transfer** by comparing raw and adapted MSE on the same held-out points. Report full-map ranking, opposite-room ranking, and accuracy for points on the same square perimeter. Also report adjacent grid edges where cost rises when legal distance decreases by at least 0.5 units.
- Render the true legal distance, raw cost, and final adapted cost for every unseen goal. This notebook measures static geometry using hand-positioned images.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RESUME_RUN_ID=None
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/geodesic_multigoal_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-geodesic-multigoal-square')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='24b13c70e9fe11a9fdd60741c51782c86c9004ae'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ARTIFACTS=Path('/content/geodesic-multigoal-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('Output:',OUTPUT)

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in [(ENCODER_DRIVE,ENCODER,ENCODER_SHA)]:
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Fit and evaluate the eight checkpoints

To resume a run, set `RESUME_RUN_ID` and keep the same numerical runtime. The code verifies and reuses completed checkpoints after encoding the images again.

In [ ]:
started=time.monotonic()
shell([sys.executable,'-m','src.mylonas_ijepa_wm.geodesic_multigoal_square',
       '--output-dir',OUTPUT,'--encoder-checkpoint',ENCODER,
       '--dino-wm-root',DINO_ROOT,'--device','cuda'],cwd=REPO)
print('Run minutes:',round((time.monotonic()-started)/60,2))
report=json.loads((OUTPUT/'comparison.json').read_text())

## Compare seen and withheld goal references

Percentages are averaged across goals. An uphill edge is a local move that reduces legal distance but increases latent cost. Opposite-room ranking compares points that are both in the room opposite the evaluated goal.

In [ ]:
import pandas as pd

stages=report['stages']
print('Fitting goals:',report['goals'][:report['seen_goals']])
print('Withheld goals:',report['goals'][report['seen_goals']:])
print('Common held-out grid points:',report['heldout_grid_points'])
counts=pd.DataFrame([s['pairs_per_training_goal'] for s in stages],
                    index=[s['band'] for s in stages],
                    columns=[str(tuple(g)) for g in report['goals'][:report['seen_goals']]])
counts.index.name='band'
counts.insert(0,'half_width',[s['half_width'] for s in stages])
print('Eligible cumulative training triples by goal')
display(counts)

rows=[]
for stage in stages:
    for item in stage['by_goal']:
        for cohort, results in item['metrics'].items():
            for method, stats in results.items():
                rows.append(dict(band=stage['band'],goal=tuple(item['goal']),status=item['status'],
                                 cohort=cohort,method=method,**stats))
table=pd.DataFrame(rows)
table[['correct','uphill']]*=100
table.to_csv(OUTPUT/'ranking_metrics.csv',index=False)
print('Mean ranking accuracy by stage on withheld goals (percent)')
summary=table[(table.status=='withheld') & table.cohort.isin(['full_map','opposite_room','same_extent'])]
display(summary.pivot_table(index='band',columns=['cohort','method'],values='correct').round(2))
print('Wrong-direction local edges by stage on withheld goals (percent)')
local=table[(table.status=='withheld') & table.cohort.isin(['local','local_opposite_room'])]
display(local.pivot_table(index='band',columns=['cohort','method'],values='uphill').round(2))
print('Final checkpoint by goal')
final=table[(table.band==stages[-1]['band']) & table.cohort.isin(['full_map','opposite_room','same_extent'])]
display(final.pivot_table(index=['status','goal'],columns=['cohort','method'],values='correct').round(2))

## Inspect the full-map cost for each unseen goal

The color scales are separate and unclipped. Compare where the cost rises and falls. The gray rectangles mark the wall, the star marks the goal, and the y-axis increases downward.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

axis=report['axis']
for i in range(report['seen_goals'],len(report['goals'])):
    goal=report['goals'][i]
    panels=[('Legal path distance',report['truth_grid'][i]),('Raw latent MSE',report['raw_grid'][i]),
            ('Final adapted MSE',stages[-1]['by_goal'][i]['grid_costs'])]
    fig,axes=plt.subplots(1,3,figsize=(13,4.2),constrained_layout=True)
    for ax,(title,values) in zip(axes,panels):
        field=np.asarray(values,dtype=float).reshape(len(axis),len(axis))
        image=ax.pcolormesh(axis,axis,field,cmap='viridis',shading='nearest',vmin=0)
        for top,bottom in [(axis[0],26),(34,axis[-1])]:
            ax.add_patch(Rectangle((29,top),6,bottom-top,facecolor='#424242'))
        ax.scatter(*goal,marker='*',c='black',edgecolor='white',s=110,zorder=4)
        ax.set(title=title,xlim=(axis[0],axis[-1]),ylim=(axis[-1],axis[0]),
               aspect='equal',xlabel='Ball x',ylabel='Ball y')
        fig.colorbar(image,ax=ax,shrink=.78)
    fig.suptitle(f'Withheld goal {tuple(goal)}')
    fig.savefig(OUTPUT/f'withheld_goal_{i-report["seen_goals"]+1:02d}.png',dpi=130)
    plt.show()
    plt.close(fig)